# 4.3 Chat Loop: Concepts Notebook

**Generative AI Foundations · Module 4**

A notebook can't easily run an endless `input()` loop, so here we **simulate** the user with a list of messages. The ideas are exactly the same as in `chat.py`.

| # | Concept |
|---|---|
| 1 | The chat cycle |
| 2 | The rules of history |
| 3 | Rolling back a failed turn |
| 4 | Leaving the chat |
| 5 | Retrying temporary errors |
| 6 | The history window |
| 7 | Commands |
| 8 | Everything together |

**Before you start:** Ollama is running and `gemma3:1b` is pulled.

## 0. Setup

In [ ]:
%pip install -q "langchain[ollama]"

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages.utils import convert_to_messages

llm = init_chat_model("ollama:gemma3:1b", temperature=0.7)
SYSTEM = {"role": "system", "content": "You are Cue, a helpful assistant. Keep answers short."}


def show(history):
    """Print the history with each message's role."""
    for message in convert_to_messages(history):
        print(f"  [{message.type:<6}] {message.text[:60]}")

---
## 1. The Chat Cycle

```text
   read message -> add to history -> send WHOLE history -> add reply -> print
        ^                                                               |
        +---------------------------------------------------------------+
```

> **Mental Model: Ping-Pong with a Scorekeeper**
> The model forgets every shot. The scorekeeper (the history list) writes each one down and reads the whole score sheet before the next shot.

One turn of the cycle is just three lines:

In [ ]:
history = [SYSTEM]

def chat_turn(history, question):
    history.append({"role": "user", "content": question})   # add the question
    reply = llm.invoke(history)                             # send the whole history
    history.append(reply)                                   # add the reply message
    return reply.text


for question in ["Hi, my name is Ali.", "What is my name?"]:   # pretend the user typed these
    print("You:", question)
    print("Cue:", chat_turn(history, question))

The history now holds every message, in order:

In [ ]:
show(history)

---
## 2. The Rules of History

| Rule | |
|---|---|
| 1 | System message first, once |
| 2 | Add the question **before** calling the model |
| 3 | Add the reply **message**, not `reply.text` |
| 4 | If the call fails, remove the question |

### Rule 3: Right vs Wrong

In [ ]:
reply = llm.invoke([SYSTEM, {"role": "user", "content": "Say hello."}])

right = [SYSTEM, {"role": "user", "content": "Say hello."}, reply]
wrong = [SYSTEM, {"role": "user", "content": "Say hello."}, reply.text]

print("RIGHT:"); show(right)
print("WRONG:"); show(wrong)

In the wrong list, the AI's words appear as `human`: the model will think **you** said them.

---
## 3. Rolling Back a Failed Turn

```text
before:        [system, user Q1, ai A1]
add question:  [system, user Q1, ai A1, user Q2]
call FAILS ->  remove Q2
after:         [system, user Q1, ai A1]          (still clean pairs)
```

> **Mental Model: The Dropped Call**
> When a call drops, you don't hang up forever. You say "sorry, you cut out" and try again.

To see a failure, we create a model that points to a port where **nothing is running**:

In [ ]:
broken_llm = init_chat_model("ollama:gemma3:1b", base_url="http://127.0.0.1:11999")

history = [SYSTEM, {"role": "user", "content": "Hi, my name is Ali."}, llm.invoke([SYSTEM, {"role": "user", "content": "Hi, my name is Ali."}])]
print("Before:"); show(history)

history.append({"role": "user", "content": "What is my name?"})
try:
    reply = broken_llm.invoke(history)
    history.append(reply)
except Exception as error:
    history.pop()   # roll back
    print(f"\nCue: Sorry, something went wrong ({type(error).__name__}).\n")

print("After:"); show(history)

The failed question was removed, so history is exactly as it was before.

---
## 4. Leaving the Chat

| User does | Python sees | We handle it by |
|---|---|---|
| Types `exit` / `quit` / `bye` | normal text | checking the text |
| Presses Enter on nothing | empty text | asking again |
| Presses Ctrl+C | `KeyboardInterrupt` | `except` around `input()` |
| Presses Ctrl+D | `EOFError` | the same `except` |

In [ ]:
EXIT_WORDS = ("exit", "quit", "bye")

for text in ["Hello", "  EXIT  ", "Bye", "", "quit please"]:
    cleaned = text.strip().lower()
    if not cleaned:
        result = "empty: ask again"
    elif cleaned in EXIT_WORDS:
        result = "leave the chat"
    else:
        result = "send to the model"
    print(f"{text!r:<16} -> {result}")

Notice `"quit please"` is **not** an exit word: only exact matches leave. Now see how Ctrl+C is caught (we raise it ourselves, as if the user pressed it):

In [ ]:
try:
    raise KeyboardInterrupt   # what Python does when you press Ctrl+C
except (KeyboardInterrupt, EOFError):
    print("Goodbye!")

---
## 5. Retrying Temporary Errors

Some failures fix themselves (a busy server, a network blip). `.with_retry()` tries again automatically.

> **Mental Model: Knocking Twice**
> If nobody answers the door, knock once more before leaving.

Here is a pretend service that fails the **first** time and works the second time:

In [ ]:
from langchain_core.runnables import RunnableLambda

attempts = {"count": 0}

def flaky_service(question):
    attempts["count"] += 1
    if attempts["count"] == 1:
        raise TimeoutError("server busy")
    return f"Answer to: {question}"

service = RunnableLambda(flaky_service)

print(service.with_retry(stop_after_attempt=2).invoke("What is Python?"))
print("Attempts made:", attempts["count"])

Without retry, the first call would have failed. With a real model you write:

```python
llm_with_retry = llm.with_retry(stop_after_attempt=2)
```

---
## 6. The History Window

History grows every turn. Sending everything costs more tokens. A **window** sends only the latest turns.

```text
full history:    [system, Q1, A1, Q2, A2, Q3, A3, Q4, A4, Q5]
sent (2 turns):  [system,             Q3, A3, Q4, A4, Q5]
```

> **Mental Model: Short-Term Memory**
> Like reading only the last few pages of your notes before a meeting. The full notebook is still kept.

In [ ]:
MAX_TURNS = 2

def recent(history):
    return history[:1] + history[1:][-(MAX_TURNS * 2 + 1):]


full = ["system", "Q1", "A1", "Q2", "A2", "Q3", "A3", "Q4", "A4", "Q5"]
print("full:", full)
print("sent:", recent(full))

The sent list always starts with the system message, then a **question** (never a reply). That is why the code uses `* 2 + 1`.

---
## 7. Commands

Commands start with `/`. They are handled **by your code**, never sent to the model.

| Command | Does |
|---|---|
| `/history` | Show the conversation |
| `/reset` | Clear memory (keep the system message) |
| `/exit` | Leave |

In [ ]:
def handle_command(text, history):
    if text == "/history":
        show(history)
    elif text == "/reset":
        history[:] = [SYSTEM]          # empty the list in place, keep the system message
        print("Memory cleared.")
    else:
        print("Unknown command.")


history = [SYSTEM, {"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello!"}]
handle_command("/history", history)
handle_command("/reset", history)
handle_command("/history", history)

---
## 8. Everything Together

A full chat session, with the user's messages simulated by a list:

In [ ]:
llm_safe = llm.with_retry(stop_after_attempt=2)
history = [SYSTEM]
user_inputs = ["Hi, my name is Ali.", "", "What is my name?", "/history", "/reset", "What is my name?", "bye"]

for text in user_inputs:
    print(f"You: {text}")
    cleaned = text.strip()

    if not cleaned:
        continue
    if cleaned.lower() in EXIT_WORDS:
        print("Goodbye!")
        break
    if cleaned.startswith("/"):
        handle_command(cleaned.lower(), history)
        continue

    history.append({"role": "user", "content": cleaned})
    try:
        reply = llm_safe.invoke(recent(history))
    except Exception as error:
        history.pop()
        print(f"Cue: Sorry, something went wrong ({type(error).__name__}).")
        continue

    history.append(reply)
    print(f"Cue: {reply.text}")

After `/reset`, Cue can no longer remember the name, because memory **is** the list.

---
## Summary

| Concept | Remember |
|---|---|
| Chat cycle | Read, add, send the whole history, add the reply, print |
| History rules | System first; question before the call; append the reply **message**; roll back on failure |
| Leaving | Exit words, empty input, `KeyboardInterrupt`, `EOFError` |
| Errors | `try` / `except` keeps the chat alive |
| Retry | `.with_retry(stop_after_attempt=2)` for temporary failures |
| Window | Send the system message + recent turns; keep the full history |
| Commands | Handled by your code, never sent to the model |

## Practice

1. In Section 8, add a `/count` command that prints how many messages are in the history.
2. In Section 6, change `MAX_TURNS` to 1 and predict what `recent(full)` returns before running it.
3. In Section 3, remove `history.pop()` and look at the history after the failure. What is wrong with it?